# Research Analysis: Signal Recurrence & Sensitivity

## 1. Introduction
This notebook documents the scientific evaluation of recurrent neural architectures (RNN and LSTM) applied to the task of extraction pure sine waves from a composite noisy signal. We focus on the frequency-dependent noise sensitivity and theoretical framework of our approach.

## 2. Theoretical Framework

### 2.1 Nyquist-Shannon Sampling Theorem
The Nyquist-Shannon sampling theorem is a fundamental bridge between continuous-time signals and discrete-time signals. It states that to perfectly reconstruct a signal with maximum frequency $f_{max}$, the sampling frequency $f_s$ must satisfy:

$$ f_s > 2 f_{max} $$

In our project, the maximum frequency component is $7\text{ Hz}$. Our sampling rate is $1000\text{ Hz}$, which is well above the Nyquist rate of $14\text{ Hz}$, ensuring no aliasing occurs.

### 2.2 LSTM Cell State Updates
The Long Short-Term Memory (LSTM) network uses gated mechanisms to maintain long-term dependencies. The cell state and hidden state updates are defined by the following equations:

1. **Forget Gate:** Decides what information to discard from the previous cell state.
   $$ f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) $$

2. **Input Gate:** Decides which new information to store in the cell state.
   $$ i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) $$

3. **Candidate Cell State:** Creates a vector of new candidate values.
   $$ \tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) $$

4. **Cell State Update:** Updates the old cell state $C_{t-1}$ into the new cell state $C_t$.
   $$ C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t $$

5. **Output Gate:** Decides what the next hidden state will be.
   $$ o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) $$

6. **Hidden State Update:**
   $$ h_t = o_t \odot \tanh(C_t) $$

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from src.sdk.analysis_utils import run_sensitivity_sweep
from src.sdk.dataset import SignalDataset
from src.models.lstm import LSTMFilter
from src.sdk.gatekeeper import APIGatekeeper

sns.set_theme(style="whitegrid")
print("Environment Ready")

## 3. Sensitivity Analysis
We investigate how increasing noise levels affect the extraction accuracy (MSE) for low-frequency (1Hz) vs. high-frequency (7Hz) signals.

In [ ]:
noise_levels = [0.05, 0.1, 0.2, 0.4, 0.6]
results = run_sensitivity_sweep(noise_levels, epochs=1)

df = pd.DataFrame({
    'Noise Level': noise_levels,
    '1Hz MSE': results[0],
    '7Hz MSE': results[3]
})

plt.figure(figsize=(10, 6))
sns.lineplot(data=pd.melt(df, ['Noise Level']), x='Noise Level', y='value', hue='variable', marker='o')
plt.title("Sensitivity Analysis: MSE vs. Noise Level")
plt.ylabel("Mean Squared Error")
plt.xlabel("Gaussian Noise Standard Deviation")
plt.show()

## 4. Signal Visualization
Comparison of the noisy input composite signal, the clean target, and the model prediction.

In [ ]:
# Generate a single sample for visualization
dataset = SignalDataset(num_samples=1, window_size=200, noise_level=0.2)
x, y = dataset[0]

model = LSTMFilter(hidden_dim=32)
gk = APIGatekeeper(model)
with torch.no_grad():
    pred = gk.run_inference(x.unsqueeze(0))

plt.figure(figsize=(12, 6))
plt.plot(x[:, 4].numpy(), label='Noisy Summed Input', alpha=0.5, color='gray')
plt.plot(y.numpy(), label='Target Pure Sine (Clean)', linewidth=2, color='blue')
plt.plot(pred[0].numpy(), label='Model Prediction', linestyle='--', color='red')
plt.title("Signal Overlap: Input vs. Target vs. Prediction")
plt.legend()
plt.show()